# Bats: test on a whole clade

Bats (order Chiroptera, taxid 9397) are one of the test cases, mainly to test getting raw data for many species at once.
This notebook lists every bat taxon that has RNA-seq runs in ENA, runs the selection (`collect_raw_data`) on every species, and counts how many species get 10 runs, fewer than 10, or none.
No reads file is downloaded, only ENA's answers (it takes a few minutes, because there is one request per species).

In [1]:
import sys
sys.path.append("../../..")

from collections import Counter
from backend.artifact_links.common import fetch
from backend.raw_data.ena_runs import SEARCH_URL
from backend.raw_data.raw_data_export import collect_raw_data

## 1. Every bat taxon with RNA-seq runs

One request to ENA, with `tax_tree(9397)`: all the runs of Chiroptera and of everything below it. We only ask for the taxid and the name of each run, then count the runs per taxon.

In [2]:
params = {
    "result": "read_run",
    "query": 'tax_tree(9397) AND library_strategy="RNA-Seq"',
    "fields": "tax_id,scientific_name",
    "limit": 0,
    "format": "json",
}
all_runs = fetch(SEARCH_URL, "list of bat runs", "ENA", params=params).json()
runs_per_taxon = Counter((run["tax_id"], run["scientific_name"]) for run in all_runs)

print(len(all_runs), "RNA-seq runs in", len(runs_per_taxon), "bat taxa")
runs_per_taxon.most_common(10)

4131 RNA-seq runs in 184 bat taxa


[(('9407', 'Rousettus aegyptiacus'), 361),
 (('9417', 'Artibeus jamaicensis'), 359),
 (('51298', 'Myotis myotis'), 220),
 (('59479', 'Rhinolophus ferrumequinum'), 202),
 (('59463', 'Myotis lucifugus'), 200),
 (('9397', 'Chiroptera'), 191),
 (('59477', 'Rhinolophus affinis'), 182),
 (('58065', 'Eonycteris spelaea'), 153),
 (('89399', 'Rhinolophus sinicus'), 144),
 (('9402', 'Pteropus alecto'), 141)]

## 2. Not every taxon is a species

Some runs are only saved at the genus level ("Myotis"), or even only as "Chiroptera", and some at the subspecies level (3 words) or as an unnamed species ("sp.").
Here a species is a name of exactly two words, without "sp.". This is a simple rule, good enough for this test.
The runs saved under the other taxa are not used, because `ena_runs` searches the exact taxid of each species.

In [3]:
def is_species(name):
    words = name.split()
    return len(words) == 2 and words[1] not in ("sp.", "cf.")

species = {key: n for key, n in runs_per_taxon.items() if is_species(key[1])}
others = {key: n for key, n in runs_per_taxon.items() if not is_species(key[1])}

print(len(species), "species with", sum(species.values()), "runs")
print(len(others), "other taxa with", sum(others.values()), "runs, for example:")
for (taxid, name), n in sorted(others.items(), key=lambda item: -item[1])[:8]:
    print(f"  {name:40} {n:4} runs")

168 species with 3835 runs
16 other taxa with 296 runs, for example:
  Chiroptera                                191 runs
  Pteropus melanotus natalis                 44 runs
  Rhinolophus macrotis episcopus             30 runs
  Rhinolophus                                10 runs
  Myotis                                      4 runs
  Hipposideros                                3 runs
  Brachyphylla nana pumila                    3 runs
  Cynopterus sp. USNM MAMM 619523             2 runs


## 3. Choose the runs of every species

One pass (RNA-Seq) for each species, with the default settings (at most 10 runs, at least 1 billion bases, no single cell or microbe runs).

In [4]:
species_list = [{"taxid": taxid, "species": name} for (taxid, name) in species]
export = collect_raw_data(species_list)
print(len(export["species"]), "species done,", len(export["problems"]), "problems")

168 species done, 0 problems


## 4. How many species get 10 runs?

In [5]:
def group(entry):
    n = len(entry["runs"])
    if n == 10:
        return "10 runs"
    if n == 0:
        return "no run"
    return "1 to 9 runs"

print(Counter(group(entry) for entry in export["species"]))

Counter({'1 to 9 runs': 130, '10 runs': 29, 'no run': 9})


## 5. Why do some species get fewer than 10 runs?

For each species with fewer than 10 runs: how many runs ENA has, how many are left after removing the small, single cell and microbe runs, and how many different studies they come from.
If a species has few useful runs or few studies, the rules stop early instead of taking replicates.

In [6]:
print(f'{"species":32} {"ENA":>5} {"useful":>6} {"studies":>7} {"chosen":>6}')
for entry in sorted(export["species"], key=lambda e: -e["total_runs"]):
    if len(entry["runs"]) < 10:
        studies = len({run["study_accession"] for run in entry["runs"]})
        print(f'{entry["species"]:32} {entry["total_runs"]:5} {entry["runs_useful"]:6} {studies:7} {len(entry["runs"]):6}')

species                            ENA useful studies chosen
Vespertilio sinensis                77     77       4      6
Tadarida brasiliensis               52     52       5      5
Pipistrellus kuhlii                 45     44       2      2
Eidolon helvum                      43     43       2      5
Barbastella barbastellus            43     43       1      1
Hipposideros pratti                 42      1       1      1
Pteropus vampyrus                   41     41       3      3
Miniopterus natalensis              38     31       2      3
Diphylla ecaudata                   38     38       1      1
Rhinolophus pusillus                34     33       6      8
Myotis pilosus                      29     15       7      7
Myotis velifer                      29     13       1      1
Otomops harrisoni                   27     25       1      1
Pteronotus quadridens               27     27       4      7
Erophylla sezekorni                 25     21       2      5
Phyllostomus discolor   

## 6. Which tissues are chosen most often?

The tissues found in the selections of all the bat species together.

In [7]:
tissues = Counter(run["tissue_type"] for entry in export["species"] for run in entry["runs"])
print(len(tissues), "different tissue names")
tissues.most_common(15)

90 different tissue names


[('heart', 61),
 ('eye', 48),
 ('liver', 47),
 (None, 47),
 ('kidney', 44),
 ('mixed', 40),
 ('brain', 31),
 ('spleen', 30),
 ('intestine', 22),
 ('lung', 21),
 ('pancreas', 14),
 ('stomach', 14),
 ('cochlea', 13),
 ('whole brain (without cerebellum)', 12),
 ('submandibular gland', 11)]

## 7. What this test shows

Only 29 of the 168 bat species get 10 runs, and 9 get none. Looking at some species with only 1 run chosen, there are several reasons:
- **Real replicates**: *Barbastella barbastellus* has 43 runs, all "blood" from one study. Choosing only 1 run is right here.
- **The tissue is not in `tissue_type`**: *Diphylla ecaudata* has 38 runs with an empty `tissue_type`, but the tissue is written in the sample title ("pancreas", "stomach"...). The script sees them all as the same tissue (None) from the same study, so it takes only 1 run, and real diversity is lost.
- **Not the kind of sample we want**: *Otomops harrisoni* has 27 "rectal and oral swab" runs from a study on cave bats and their viruses.
- **Wrong data type in ENA**: *Myotis velifer* has 29 runs marked RNA-Seq, but their titles say PRO-seq, a different technique, done on cells modified in the lab.

Also, 296 runs (7%) are saved under a genus ("Myotis"), a subspecies ("Pteropus melanotus natalis") or only "Chiroptera" (191 runs), so they are never used, because each species is searched with its exact taxid.
